## WESAD — ¿funciona en la muñeca el estimador elegido en el dedo (CASE)?
**Prueba externa, sin ajustar nada:** se aplican a la PPG de muñeca (Empatica E4, 64 Hz) los mismos métodos y
parámetros del notebook 14. El método principal es **Count** (elegido en CASE); no se elige nada con WESAD.

- **Referencia de RR:** banda respiratoria del pecho (RespiBAN, 700 Hz), con el mismo control de fiabilidad que en
  CASE (conteo y FFT de la banda coinciden ≤ 2 rpm).
- **Control de latidos:** la FC de la PPG de muñeca se compara con la del ECG del pecho. Si no coinciden, el
  problema está en detectar los latidos, no en la respiración.
- **Utilidad:** detección de **estrés** (TSST) frente a no estrés (línea base y diversión), dejando un participante
  fuera, con HRV ± RR de la banda o de la PPG. Cautela: en el TSST se habla, lo que altera la respiración.
- Ventanas de 30 s (paso 10 s) dentro de una sola condición; 60 s cambiando `VENTANA`.

In [7]:
# w-1 — Imports, configuración y rutas
import json
import pickle
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, filtfilt, find_peaks, resample_poly
from scipy.stats import wilcoxon, false_discovery_control
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR / "notebooks"))
sys.path.insert(0, str(Path.cwd()))
import rr_generalizable as rg

VENTANA = 30                                  # s. Principal: 30. Sensibilidad: 60.
rg.VENTANA_S, rg.N = VENTANA, VENTANA * rg.FS
OUT = BASE_DIR / "results"
RES = OUT / f"wesad_{VENTANA}s"
RES.mkdir(parents=True, exist_ok=True)
CFG = json.loads((OUT / "config_congelada.json").read_text())
P = {**rg.PARAMS, "rpm_min": CFG["rpm_min"], "rpm_max": CFG["rpm_max"], "lp_riiv_hz": CFG["lp_fft_hz"],
     "lp_art_hz": CFG["lp_art_hz"], "umbral_art": CFG["umbral_art"]}
PASO_S, FS = rg.PASO_S, rg.FS
FS_PECHO, FS_BVP = 700, 64
WESAD_DIR = next((p.parent.parent for p in (BASE_DIR / "data" / "WESAD").rglob("S2.pkl")), BASE_DIR / "data" / "WESAD")
print("WESAD en:", WESAD_DIR)
SUJETOS = sorted((p.name for p in WESAD_DIR.glob("S*") if (p / f"{p.name}.pkl").exists()), key=lambda s: int(s[1:]))
CONDICION = {1: "línea base", 2: "estrés", 3: "diversión"}      # 4 = meditación (no se usa)
ESTRES = {1: 0, 2: 1, 3: 0}
INVERTIR = False                              # cambiar a True si el control de FC (w-4) indica pulso invertido
pd.set_option("display.width", 250)
warnings.filterwarnings("ignore", category=RuntimeWarning)
print(f"{len(SUJETOS)} sujetos | ventana {VENTANA} s, paso {PASO_S} s | resultados en {RES}")

WESAD en: c:\Users\Usuario\Documents\respiratory-rate-ppg\data\WESAD
15 sujetos | ventana 30 s, paso 10 s | resultados en c:\Users\Usuario\Documents\respiratory-rate-ppg\results\wesad_30s


In [8]:
# w-2 — Funciones: referencia de la banda, latidos del ECG, HRV
def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def a_25hz(x, fs):
    return decimate(decimate(x, 7, zero_phase=True), 4, zero_phase=True) if fs == 700 else resample_poly(x, 25, fs)

def marcas_banda(resp):
    """Inhalaciones en la banda del pecho: mismo criterio que en CASE (25 Hz, 0.05–1 Hz, ≥ 1 s, 0.25·RIC)."""
    x = a_25hz(resp, FS_PECHO)
    b, a = butter(2, [0.05, 1.0], btype="band", fs=25)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=25, prominence=0.25 * iqr)
    return pk / 25, x

def latidos_ecg(ecg, fs=FS_PECHO):
    x = decimate(ecg, 4, zero_phase=True)                  # 175 Hz
    f = fs / 4
    b, a = butter(2, [5, 20], btype="band", fs=f)
    e = np.abs(filtfilt(b, a, x))
    pk, _ = find_peaks(e, distance=int(0.33 * f), height=np.percentile(e, 98) * 0.35)
    t = pk / f
    ibi = np.diff(t)
    t = t[1:]
    med = pd.Series(ibi).rolling(11, center=True, min_periods=3).median().to_numpy()
    ok = (ibi >= 0.33) & (ibi <= 1.5) & (np.abs(ibi - med) <= 0.3 * med)
    return t[ok], ibi[ok]

def hrv(t, ibi, t0):
    m = (t >= t0) & (t < t0 + VENTANA)
    if m.sum() < 15:
        return np.nan, np.nan, np.nan
    tw, iw = t[m], ibi[m]
    consec = np.abs(np.diff(tw) - iw[1:]) < 0.02
    d = np.diff(iw)[consec]
    rmssd = np.log(1000 * np.sqrt(np.mean(d ** 2))) if consec.sum() >= 10 else np.nan
    return 60 / iw.mean(), rmssd, np.log(1000 * iw.std())

def maximo(x, lo, hi=P["rpm_max"]):
    f, Pw = rg.espectro(x, rg.FS_M, 1024)
    b = (f >= lo) & (f <= hi)
    return f[b][np.argmax(Pw[b])]

In [9]:
# w-3 — Ventanas y estimaciones (tarda unos minutos)
filas = []
for s in SUJETOS:
    with open(WESAD_DIR / s / f"{s}.pkl", "rb") as fh:
        d = pickle.load(fh, encoding="latin1")
    resp = np.asarray(d["signal"]["chest"]["Resp"], float).ravel()
    ecg = np.asarray(d["signal"]["chest"]["ECG"], float).ravel()
    bvp = np.asarray(d["signal"]["wrist"]["BVP"], float).ravel()
    lab = np.asarray(d["label"]).ravel()
    marcas, resp25 = marcas_banda(resp)
    t_e, ibi_e = latidos_ecg(ecg)
    est = rg.Estimador(bvp, FS_BVP, P, invertir=INVERTIR)
    dur = min(est.duracion, len(lab) / FS_PECHO, len(resp) / FS_PECHO)
    for t0 in np.round(np.arange(0, dur - VENTANA + 1e-9, PASO_S), 3):
        l = lab[int(t0 * FS_PECHO):int((t0 + VENTANA) * FS_PECHO)]
        if len(l) == 0 or l.min() != l.max() or l[0] not in CONDICION:
            continue
        r = rr_ref(marcas, t0, t0 + VENTANA)
        seg = resp25[int(t0 * 25):int((t0 + VENTANA) * 25)]
        if not np.isfinite(r) or not P["rpm_min"] <= r <= P["rpm_max"] or len(seg) < VENTANA * 25:
            continue
        x5 = decimate(seg, 5, zero_phase=True)            # 5 Hz para la FFT de la banda
        f, Pw = rg.espectro(x5, 5, 2048)
        bnd = (f >= P["rpm_min"]) & (f <= P["rpm_max"])
        if abs(r - f[bnd][np.argmax(Pw[bnd])]) > 2:
            continue                                        # referencia no fiable
        w = est.ventana(t0)
        series = [rg.serie_4hz(est.t, x, t0) for x in est.mods.values()]
        ok = [x for x in series if x is not None]
        sf = [maximo(x, P["rpm_min"]) for x in ok]
        fc, rmssd, sdnn = hrv(est.t, est.mods["RIFV"], t0)
        fc_e = hrv(t_e, ibi_e, t0)[0]
        filas.append({"Sujeto": s, "Inicio (s)": t0, "Condición": CONDICION[int(l[0])],
                      "Estrés": ESTRES[int(l[0])], "RR banda": r, "Artefacto": w["Artefacto"],
                      "FFT-std": w["A"], "SmartFusion": np.mean(sf) if len(sf) == 3 and np.std(sf) <= 4 else np.nan,
                      "FFT-1/f": w["S"], "Count": w["D"], "Fusion": w["F"],
                      "RR por modulación": {k: (rg.conteo(x, P["hp_fijo_hz"], P) if x is not None else np.nan)
                                            for k, x in zip(est.mods, series)},
                      "FC": fc, "lnRMSSD": rmssd, "lnSDNN": sdnn, "FC ECG": fc_e})
    print(s, end=" ")
R = pd.DataFrame(filas)
for k in ["RIIV", "RIAV", "RIFV"]:
    R[f"Count {k}"] = R["RR por modulación"].map(lambda m: m[k])
R = R.drop(columns="RR por modulación")
R.to_csv(RES / "wesad_ventanas.csv", index=False)
V = R[~R["Artefacto"]].copy()
print(f"\n{len(R)} ventanas con referencia fiable; {len(V)} sin artefacto")
print(V.groupby("Condición").size().to_string())

S2 S3 S4 S5 S6 S7 S8 S9 S10 S11 S13 S14 S15 S16 S17 
2069 ventanas con referencia fiable; 1108 sin artefacto
Condición
diversión     274
estrés        101
línea base    733


In [10]:
# w-4 — Control: ¿se detectan bien los latidos en la muñeca? (FC PPG frente a FC ECG)
e = (V["FC"] - V["FC ECG"]).abs()
print(f"Error de FC (PPG muñeca vs ECG): mediana {e.median():.2f} lpm | % ≤ 5 lpm: {100 * (e <= 5).mean():.0f} %")
print(e.groupby(V["Sujeto"]).median().round(1).to_string())
print("\nSi el error es grande (> 5 lpm) en muchos sujetos, prueba INVERTIR = True en w-1 y vuelve a correr.")

Error de FC (PPG muñeca vs ECG): mediana 0.97 lpm | % ≤ 5 lpm: 91 %
Sujeto
S10    0.7
S11    0.9
S13    1.1
S14    0.7
S15    0.9
S16    1.8
S17    1.0
S2     1.1
S3     1.1
S4     1.0
S5     2.5
S6     1.8
S7     0.9
S8     0.9
S9     0.5

Si el error es grande (> 5 lpm) en muchos sujetos, prueba INVERTIR = True en w-1 y vuelve a correr.


In [11]:
# w-5 — Exactitud de la RR (todos los sujetos; prueba externa, sin elegir nada)
def metricas(g, col):
    ok = g[col].notna()
    if ok.sum() == 0:
        return {"Cobertura %": 0.0}
    h = g[ok]
    err = h[col] - h["RR banda"]
    e = err.abs()
    r = h.groupby("Sujeto").apply(lambda q: q[col].corr(q["RR banda"]) if q[col].std() > 0 else np.nan)
    dl = h.groupby(["Sujeto", "Estrés"])[[col, "RR banda"]].mean().unstack()
    if (col, 1) in dl and (col, 0) in dl:
        dm, dr = dl[(col, 1)] - dl[(col, 0)], dl[("RR banda", 1)] - dl[("RR banda", 0)]
    else:
        dm = dr = pd.Series(dtype=float)
    return {"MAE mediano": e.groupby(h["Sujeto"]).mean().median(), "RMSE": np.sqrt((err ** 2).mean()),
            "Sesgo": err.mean(), "LdA inf": err.mean() - 1.96 * err.std(), "LdA sup": err.mean() + 1.96 * err.std(),
            "% ≤ 2 rpm": 100 * (e <= 2).mean(), "Cobertura %": 100 * ok.mean(),
            "r dentro de sujeto": r.median(), "Δ estrés método": dm.median(), "Δ estrés banda": dr.median()}

METODOS = ["FFT-std", "SmartFusion", "FFT-1/f", "Count", "Fusion", "Count RIIV", "Count RIAV", "Count RIFV"]
V["Control: media del sujeto (oráculo)"] = V.groupby("Sujeto")["RR banda"].transform("mean")
TAB = pd.DataFrame({m: metricas(V, m) for m in METODOS + ["Control: media del sujeto (oráculo)"]}).T
TAB.to_csv(RES / "wesad_resumen.csv")
print(TAB.round(2).to_string())
print("\nReferencia en dedo (CASE, prueba, 30 s): Count MAE 1.01 | FFT-std 8.88 | Fusion 0.62")

                                     MAE mediano  RMSE  Sesgo  LdA inf  LdA sup  % ≤ 2 rpm  Cobertura %  r dentro de sujeto  Δ estrés método  Δ estrés banda
FFT-std                                     4.87  6.69   0.72   -12.33    13.77      36.64       100.00                0.05             2.60           -0.26
SmartFusion                                 6.72  8.11  -6.75   -15.57     2.07      15.62        57.76               -0.04            -1.81            0.49
FFT-1/f                                     3.49  5.13   0.91    -8.98    10.80      50.14        98.83                0.06             2.03           -0.38
Count                                       2.56  3.27   0.55    -5.76     6.86      55.16        98.83                0.17             1.15           -0.38
Fusion                                      2.54  4.09   0.65    -7.28     8.58      62.39        59.75                0.15             2.88           -0.08
Count RIIV                                  3.48  4.71   1

In [12]:
# w-6 — Detección de estrés (dejando un sujeto fuera; 3 clasificadores)
MODELOS = {"Regresión logística": lambda: LogisticRegression(max_iter=1000),
           "Random forest": lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
           "Gradient boosting": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
                                                                       random_state=0)}
BRAZOS = {"HRV": ["FC", "lnRMSSD", "lnSDNN"],
          "HRV + RR banda": ["FC", "lnRMSSD", "lnSDNN", "RR banda"],
          "HRV + Count": ["FC", "lnRMSSD", "lnSDNN", "Count"],
          "HRV + FFT-std": ["FC", "lnRMSSD", "lnSDNN", "FFT-std"]}
cols = sorted({c for cs in BRAZOS.values() for c in cs})
C = V.dropna(subset=cols).copy()
C[cols] = C.groupby("Sujeto")[cols].transform(lambda c: (c - c.mean()) / (c.std() + 1e-9))
y, part = C["Estrés"].to_numpy(), C["Sujeto"].to_numpy()
print(f"Ventanas: {len(C)} | sujetos: {len(np.unique(part))} | % estrés: {100 * y.mean():.0f} %")
filas = []
for mod, crear in MODELOS.items():
    for brazo, cs in BRAZOS.items():
        X = C[cs].to_numpy()
        for s in np.unique(part):
            te = part == s
            if len(np.unique(y[te])) < 2:
                continue
            pr = crear().fit(X[~te], y[~te]).predict_proba(X[te])[:, 1]
            filas.append({"Modelo": mod, "Brazo": brazo, "Sujeto": s, "AUC": roc_auc_score(y[te], pr)})
A = pd.DataFrame(filas)
A.to_csv(RES / "wesad_auc.csv", index=False)
print(A.pivot_table(index="Modelo", columns="Brazo", values="AUC", aggfunc="median")[list(BRAZOS)].round(3).to_string())
filas = []
for mod, g in A.groupby("Modelo", sort=False):
    W = g.pivot(index="Sujeto", columns="Brazo", values="AUC")
    for a, b in [("HRV + RR banda", "HRV"), ("HRV + Count", "HRV"), ("HRV + FFT-std", "HRV"),
                 ("HRV + Count", "HRV + RR banda"), ("HRV + Count", "HRV + FFT-std")]:
        dd = (W[a] - W[b]).dropna()
        filas.append({"Modelo": mod, "Comparación": f"{a} vs {b}", "Δ AUC mediana": dd.median(),
                      "p": wilcoxon(dd).pvalue if (dd != 0).any() and len(dd) > 1 else np.nan})
CMP = pd.DataFrame(filas)
CMP["p ajustada (BH)"] = false_discovery_control(CMP["p"].fillna(1).to_numpy())
CMP.to_csv(RES / "wesad_comparaciones.csv", index=False)
print(CMP.round(3).to_string(index=False))

Ventanas: 1092 | sujetos: 15 | % estrés: 8 %
Brazo                  HRV  HRV + RR banda  HRV + Count  HRV + FFT-std
Modelo                                                                
Gradient boosting    0.969           0.970        0.953          0.961
Random forest        0.945           0.963        0.958          0.938
Regresión logística  0.993           0.995        0.992          0.994
             Modelo                   Comparación  Δ AUC mediana     p  p ajustada (BH)
Regresión logística         HRV + RR banda vs HRV          0.000 0.917            0.917
Regresión logística            HRV + Count vs HRV          0.000 0.575            0.785
Regresión logística          HRV + FFT-std vs HRV          0.000 0.499            0.785
Regresión logística HRV + Count vs HRV + RR banda          0.000 0.260            0.785
Regresión logística  HRV + Count vs HRV + FFT-std          0.000 0.515            0.785
      Random forest         HRV + RR banda vs HRV         -0.002 0.534  